# Uzak Bir Galaksinin Parlaklık Analizi (MCMC)

Bu çalışma, gürültülü gözlem verilerinden bir gök cisminin gerçek parlaklığını ve veri setindeki belirsizliği (standart sapma) Bayesyen yöntemlerle tahmin etmeyi amaçlar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import emcee
import corner

print("Kütüphaneler yüklendi.")

## 1. Veri Oluşturma (Sentetik Gözlem)

Gerçek değerleri simüle ederek teleskobun üreteceği gürültülü veriyi oluşturuyoruz.

In [ ]:
np.random.seed(42)
true_mu = 150.0      # Gerçek parlaklık
true_sigma = 10.0    # Gözlem hatası
n_obs = 50           # Gözlem sayısı

# Gürültülü veri oluşturma
data = true_mu + true_sigma * np.random.randn(n_obs)

plt.hist(data, bins=15, color='skyblue', edgecolor='black', alpha=0.7)
plt.axvline(true_mu, color='red', linestyle='--', label=f'Gerçek Mu ({true_mu})')
plt.title("Gözlemlenen Parlaklık Verileri Dağılımı")
plt.legend()
plt.show()

## 2. Bayesyen Fonksiyonların Tanımlanması

In [ ]:
def log_likelihood(theta, data):
    mu, sigma = theta
    if sigma <= 0: 
        return -np.inf
    return -0.5 * np.sum(((data - mu) / sigma)**2 + np.log(2 * np.pi * sigma**2))

def log_prior(theta):
    mu, sigma = theta
    if 0 < mu < 300 and 0 < sigma < 50:
        return 0.0
    return -np.inf

def log_probability(theta, data):
    lp = log_prior(theta)
    if not np.isfinite(lp):
        return -np.inf
    return lp + log_likelihood(theta, data)

## 3. MCMC Örnekleyiciyi Çalıştırma

In [ ]:
initial = [140, 5]
n_walkers = 32
ndim = 2
pos = initial + 1e-4 * np.random.randn(n_walkers, ndim)

sampler = emcee.EnsembleSampler(n_walkers, ndim, log_probability, args=(data,))
sampler.run_mcmc(pos, 2000, progress=True)

# Burn-in: 500 adımı atıyoruz
flat_samples = sampler.get_chain(discard=500, thin=15, flat=True)

## 4. Sonuçların Görselleştirilmesi (Corner Plot)

In [ ]:
fig = corner.corner(
    flat_samples, 
    labels=[r"$\mu$ (Parlaklık)", r"$\sigma$ (Hata)"],
    truths=[true_mu, true_sigma],
    show_titles=True,
    title_fmt=".2f"
)
plt.show()

## 5. İstatistiksel Analiz Tablosu

| Parametre | Gerçek Değer | Tahmin (Median) | Alt Sınır (%16) | Üst Sınır (%84) | Mutlak Hata |
|-----------|--------------|-----------------|-----------------|-----------------|-------------|
| $\mu$ | 150.0 | 147.79 | 146.43 | 149.07 | 2.21 |
| $\sigma$ | 10.0 | 9.49 | 8.55 | 10.53 | 0.51 |

## 6. Ek Analiz Soruları

### 6.1. Prior Etkisi
**Soru:** Eğer parlaklık için çok dar bir prior seçseydiniz (örneğin 100-110 arası), sonuç nasıl değişirdi?

**Yanıt:** Eğer mu için [100, 110] gibi gerçek değerden uzak bir aralık seçilseydi, model gerçek değeri (150) asla bulamazdı. Posterior dağılımı 110 sınırına yığılırdı. Bu durum, yanlış bir ön bilginin (prior) veriyi nasıl domine edebileceğini gösterir.

### 6.2. Veri Miktarı Etkisi
**Soru:** Gözlem sayısını (n_obs) 5'e düşürdüğünüzde posterior dağılımının genişliği (belirsizlik) nasıl etkileniyor?

**Yanıt:** Gözlem sayısı azaldığında istatistiksel belirsizlik artar. 5 gözlem durumunda posterior dağılımları çok daha geniş olurdu; bu da tahmin hassasiyetinin (precision) azaldığı anlamına gelir.